In [1]:
import json
import requests

# --- Wialon token ---
BASE = "https://hst-api.wialon.com/wialon/ajax.html"

TOKEN = "338f417f5f8ae25ba6eb01c878134153FE905CC0A125F5E827810F937A192125F15C8769"

RESOURCE_ID = 29702184
TEMPLATE_IDS = [31]
FLAGS = 0x1


def wialon_call(svc: str, params: dict, sid: str | None = None):
    """POST svc + JSON params; returns parsed JSON (dict or list)."""
    body = {"svc": svc, "params": json.dumps(params, separators=(",", ":"))}
    if sid:
        body["sid"] = sid
    r = requests.post(BASE, data=body, timeout=120)
    r.raise_for_status()
    return r.json()


def has_api_error(payload) -> bool:
    return isinstance(payload, dict) and payload.get("error") is not None


# 1) Login with token
login = wialon_call("token/login", {"token": TOKEN})
if has_api_error(login):
    raise RuntimeError(f"token/login failed: {login}")

sid = login["eid"]
print("Logged in, sid:", sid[:16], "...")

# 2) Load report template definition(s) for this resource
report_templates = wialon_call(
    "report/get_report_data",
    {"itemId": RESOURCE_ID, "col": TEMPLATE_IDS, "flags": FLAGS},
    sid=sid,
)

if has_api_error(report_templates):
    raise RuntimeError(f"report/get_report_data failed: {report_templates}")

# Success: list of template objects (one per id in TEMPLATE_IDS)
if not isinstance(report_templates, list):
    raise TypeError(f"Unexpected response type: {type(report_templates)}")

print(f"Received {len(report_templates)} template object(s).")
print(json.dumps(report_templates, indent=2))

# 3) Optional: end session
logout = wialon_call("core/logout", {}, sid=sid)
if has_api_error(logout):
    print("core/logout warning:", logout)
else:
    print("Logged out.")

Logged in, sid: 10ac972842dd6546 ...
Received 1 template object(s).
[
  {
    "id": 31,
    "n": "Overland - Unit FreeWheeling Driving Report",
    "ct": "avl_unit",
    "p": "{\"bind\":{\"avl_unit\":[]},\"descr\":\"\"}",
    "bsfl": {
      "ct": 1759759346,
      "mt": 1759759698
    }
  }
]
core/logout warning: {'error': 0}


### Getting all the data from track3 Echo driving template against Finished goods

In [2]:
import time
from datetime import datetime, timedelta, timezone

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

# --- Wialon ---
BASE = "https://hst-api.wialon.com/wialon/ajax.html"
TOKEN = "48c85c06ad33546358b6711224d80a3e0FEE2BA501FB996B3DDD0FFCC0DB48AA31C66ED7"

# WIALON_RESOURCE_ID="29702184"
# WIALON_TEMPLATE_ID="31"
# WIALON_UNIT_GROUP_ID="29702186"


RESOURCE_ID = 29702184
TEMPLATE_ID = 31
UNIT_GROUP_ID = 29702186
# --- Date: full calendar day 9 April (edit YEAR if needed) ---
YEAR = 2026
MONTH = 4
DAY = 9
# Whole day in UTC
start = datetime(YEAR, MONTH, DAY, 0, 0, 0, tzinfo=timezone.utc)
end = datetime(YEAR, MONTH, DAY, 23, 59, 59, tzinfo=timezone.utc)
# Optional: local offset instead of UTC (example Nairobi UTC+3)
# TZ = timezone(timedelta(hours=3))
# start = datetime(YEAR, MONTH, DAY, 0, 0, 0, tzinfo=TZ)
# end = datetime(YEAR, MONTH, DAY, 23, 59, 59, tzinfo=TZ)
TIME_FROM = int(start.timestamp())
TIME_TO = int(end.timestamp())
POLL_SEC = 180
SLEEP_SEC = 0.5
def wialon_call(svc: str, params: dict, sid: str | None = None):
    body = {"svc": svc, "params": json.dumps(params, separators=(",", ":"))}
    if sid:
        body["sid"] = sid
    r = requests.post(BASE, data=body, timeout=120)
    r.raise_for_status()
    return r.json()
def api_error(payload) -> bool:
    return isinstance(payload, dict) and payload.get("error") not in (None, 0)
def report_status_code(st) -> int | None:
    """Wialon may return status as int or str (e.g. 4 or '4')."""
    if not isinstance(st, dict) or "status" not in st:
        return None
    try:
        return int(st["status"])
    except (TypeError, ValueError):
        return None
def cell_to_text(cell):
    if isinstance(cell, dict):
        if "t" in cell:
            return cell["t"]
        if "v" in cell:
            return cell["v"]
        return json.dumps(cell, ensure_ascii=False)
    return cell
def rows_json_to_dataframe(headers, row_objects) -> pd.DataFrame:
    headers = headers or []
    n = len(headers)
    rows_out = []
    for row in row_objects:
        cells = row.get("c", []) if isinstance(row, dict) else []
        rec = [cell_to_text(cells[i]) if i < len(cells) else None for i in range(n)]
        rows_out.append(rec)
    return pd.DataFrame(rows_out, columns=headers)
def iter_leaf_rows(nodes):
    if not nodes:
        return
    for node in nodes:
        subs = node.get("r") or []
        if subs:
            yield from iter_leaf_rows(subs)
        else:
            yield node
def report_table_to_dataframe(sid, table_meta: dict, table_index: int) -> pd.DataFrame:
    headers = table_meta.get("header") or []
    nrows = int(table_meta.get("rows") or 0)
    level = int(table_meta.get("level") or 1)
    if nrows <= 0:
        return pd.DataFrame(columns=headers)
    if level == 1:
        raw = wialon_call(
            "report/get_result_rows",
            {"tableIndex": table_index, "indexFrom": 0, "indexTo": nrows - 1},
            sid=sid,
        )
        if api_error(raw):
            raise RuntimeError(raw)
        if not isinstance(raw, list):
            raise TypeError(f"Unexpected get_result_rows payload: {type(raw)}")
        return rows_json_to_dataframe(headers, raw)
    sel_level = max(level - 1, 0)
    raw = wialon_call(
        "report/select_result_rows",
        {
            "tableIndex": table_index,
            "config": {
                "type": "range",
                "data": {"from": 0, "to": nrows - 1, "level": sel_level},
            },
        },
        sid=sid,
    )
    if api_error(raw):
        raise RuntimeError(raw)
    if not isinstance(raw, list):
        raise TypeError(f"Unexpected select_result_rows payload: {type(raw)}")
    leaf_rows = list(iter_leaf_rows(raw))
    return rows_json_to_dataframe(headers, leaf_rows)
# --- Login ---
login = wialon_call("token/login", {"token": TOKEN})
if api_error(login):
    raise RuntimeError(login)
sid = login["eid"]
wialon_call("report/cleanup_result", {}, sid=sid)
exec_params = {
    "reportResourceId": RESOURCE_ID,
    "reportTemplateId": TEMPLATE_ID,
    "reportObjectId": UNIT_GROUP_ID,
    "reportObjectSecId": 0,
    "interval": {"flags": 0, "from": TIME_FROM, "to": TIME_TO},
    "remoteExec": 1,
}
ex = wialon_call("report/exec_report", exec_params, sid=sid)
if api_error(ex):
    raise RuntimeError(ex)
# --- Wait until report is ready (status 4 as int or str) ---
t0 = time.time()
while True:
    st = wialon_call("report/get_report_status", {}, sid=sid)
    code = report_status_code(st)
    if code == 4:
        break
    if time.time() - t0 > POLL_SEC:
        raise TimeoutError(
            f"Report not ready after {POLL_SEC}s: last payload {st!r}"
        )
    time.sleep(SLEEP_SEC)
applied = wialon_call("report/apply_report_result", {}, sid=sid)
if api_error(applied):
    raise RuntimeError(applied)
report_result = applied.get("reportResult") or {}
tables_meta = report_result.get("tables") or []
dfs = {}
for idx, tbl in enumerate(tables_meta):
    name = tbl.get("label") or tbl.get("name") or f"table_{idx}"
    dfs[name] = report_table_to_dataframe(sid, tbl, idx)
for name, dfx in dfs.items():
    print(f"\n=== {name} — {len(dfx)} rows ===")
    display(dfx)
df = next(iter(dfs.values())) if dfs else pd.DataFrame()
logout = wialon_call("core/logout", {}, sid=sid)
if api_error(logout):
    print("Logout note:", logout)
print(f"\nInterval: {start.isoformat()} … {end.isoformat()}")

c:\Users\MULINGWA STEPHEN\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


TimeoutError: Report not ready after 180s: last payload {'status': '8'}

### Drilling deep into echo driving so as to simplify the data to understandable manner

In [ ]:
import re
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

# ========= tuning =========
MERGE_GAP_SEC = 12
JUNK_FILTER = True
MIN_DURATION_SEC = 3
MAX_ZERO_KM_DURATION = 4

WRITE_XLSX = True
OUT_PATH = Path.home() / "Downloads" / f"eco_driving_clean_{datetime.now():%Y%m%d_%H%M%S}.xlsx"

WANT = [
    "Grouping",
    "Violation",
    "Beginning",
    "Initial location",
    "End",
    "Avg. speed",
    "Max. speed",
    "Duration",
    "Mileage",
    "Count",
    "Driver",
]


def match_col(df, want):
    w = want.strip().lower()
    for c in df.columns:
        if str(c).strip().lower() == w:
            return c
    return None


def _parse_dt(x):
    if pd.isna(x):
        return pd.NaT
    s = str(x).strip()
    if s in {"", "-", "–", "—"}:
        return pd.NaT
    for fmt in ("%d.%m.%Y %H:%M:%S", "%Y-%m-%d %H:%M:%S"):
        try:
            return pd.to_datetime(s, format=fmt)
        except ValueError:
            continue
    return pd.to_datetime(s, errors="coerce")


def _parse_dur_sec(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if s in {"", "-", "–"}:
        return np.nan
    p = s.split(":")
    try:
        if len(p) == 3:
            return float(p[0]) * 3600 + float(p[1]) * 60 + float(p[2])
        if len(p) == 2:
            return float(p[0]) * 60 + float(p[1])
    except ValueError:
        pass
    return np.nan


def _fmt_dur(sec):
    if pd.isna(sec):
        return ""
    sec = int(round(sec))
    h, r = divmod(sec, 3600)
    m, s = divmod(r, 60)
    return f"{h}:{m:02d}:{s:02d}"


def _parse_km(x):
    if pd.isna(x):
        return np.nan
    s = str(x).lower().replace("km", "").replace(",", ".").strip()
    if s in {"", "-", "–"}:
        return np.nan
    m = re.search(r"[-+]?\d*\.?\d+", s)
    return float(m.group()) if m else np.nan


def _parse_spd(x):
    if pd.isna(x):
        return np.nan
    s = str(x).lower().replace("km/h", "").replace(",", ".").strip()
    if s in {"", "-", "–"}:
        return np.nan
    m = re.search(r"[-+]?\d*\.?\d+", s)
    return float(m.group()) if m else np.nan


def _row_end_ts(row) -> pd.Timestamp:
    b = row["_b"]
    e = row["_e"]
    if pd.notna(e):
        return e
    if pd.notna(b) and pd.notna(row["_ds"]):
        return b + timedelta(seconds=float(row["_ds"]))
    return b


def _build_summary(detail: pd.DataFrame) -> pd.DataFrame:
    d = detail.copy()
    use = [c for c in d.columns if c not in ("raw_rows", "duration_sum_sec", "duration_wall_sec", "max_gap_sec")]
    d = d[[c for c in use if c in d.columns]]
    ncount = pd.to_numeric(d["Count"], errors="coerce").fillna(1)
    pt = (
        d.assign(_n=ncount)
        .pivot_table(index="Grouping", columns="Violation", values="_n", aggfunc="sum", fill_value=0)
        .reset_index()
    )
    vcols = [c for c in pt.columns if c != "Grouping"]
    pt["violations_count"] = pt[vcols].sum(axis=1)
    if "Driver" in d.columns:
        pt["Driver"] = d.groupby("Grouping", dropna=False)["Driver"].first().reindex(pt["Grouping"]).values
    return pt.sort_values("violations_count", ascending=False)


def _merge_bursts(df: pd.DataFrame) -> pd.DataFrame:
    out = []
    for (g, v), sub in df.groupby(["Grouping", "Violation"], dropna=False):
        sub = sub.sort_values("_b")
        chunk = []
        prev_end = pd.NaT

        def flush():
            nonlocal chunk, prev_end
            if not chunk:
                return
            ch = df.loc[chunk].sort_values("_b")
            first, last = ch.iloc[0], ch.iloc[-1]
            b0 = first["_b"]
            e_last = _row_end_ts(last)
            if pd.isna(e_last):
                e_last = b0

            miles = ch["_mk"].sum(min_count=1)
            cnt = pd.to_numeric(ch["Count"], errors="coerce").fillna(1).sum()
            dsum = ch["_ds"].sum(min_count=1)
            mx = ch["Max. speed"].map(_parse_spd).max()

            gaps = []
            for i in range(1, len(ch)):
                pe = _row_end_ts(ch.iloc[i - 1])
                nb = ch.iloc[i]["_b"]
                if pd.notna(pe) and pd.notna(nb):
                    gaps.append((nb - pe).total_seconds())
            max_gap = max(gaps) if gaps else 0.0

            wall_sec = np.nan
            if pd.notna(b0) and pd.notna(e_last):
                wall_sec = (e_last - b0).total_seconds()

            drv_series = ch["Driver"].dropna()
            drv = drv_series.iloc[0] if len(drv_series) else first["Driver"]

            out.append(
                {
                    "Grouping": g,
                    "Violation": v,
                    "Beginning": first["Beginning"],
                    "Initial location": first["Initial location"],
                    "End": last["End"],
                    "Avg. speed": first["Avg. speed"],
                    "Max. speed": f"{mx:.0f} km/h" if pd.notna(mx) else first["Max. speed"],
                    "Duration": _fmt_dur(dsum) if pd.notna(dsum) else "",
                    "Mileage": f"{miles:.2f} km" if pd.notna(miles) else "",
                    "Count": int(cnt),
                    "Driver": drv,
                    "raw_rows": len(chunk),
                    "duration_sum_sec": float(dsum) if pd.notna(dsum) else np.nan,
                    "duration_wall_sec": float(wall_sec) if pd.notna(wall_sec) else np.nan,
                    "max_gap_sec": float(max_gap),
                }
            )
            chunk = []

        for i in sub.index:
            row = df.loc[i]
            b = row["_b"]
            e_eff = _row_end_ts(row)

            if not chunk:
                chunk = [i]
                prev_end = e_eff
                continue

            gap = (b - prev_end).total_seconds() if pd.notna(b) and pd.notna(prev_end) else np.inf
            if gap <= MERGE_GAP_SEC:
                chunk.append(i)
                prev_end = e_eff
            else:
                flush()
                chunk = [i]
                prev_end = e_eff
        flush()

    return pd.DataFrame(out)


# ========= 1) Eco → eco_detail =========
eco = next(
    (df.copy() for lab, df in dfs.items() if re.search(r"eco", str(lab), re.I)),
    None,
)
if eco is None:
    raise ValueError("No eco table in dfs: " + repr(list(dfs.keys())))

eco.columns = [str(c).strip() for c in eco.columns]
colmap = {w: match_col(eco, w) for w in WANT}
present = [w for w in WANT if colmap.get(w)]
eco_detail = eco[[colmap[w] for w in present]].rename(columns={colmap[w]: w for w in present})

# ========= 2) Raw + clean =========
eco_detail_raw = eco_detail.copy()

work = eco_detail_raw.copy()
work["_b"] = work["Beginning"].map(_parse_dt)
work["_e"] = work["End"].map(_parse_dt)
work["_ds"] = work["Duration"].map(_parse_dur_sec)
work["_mk"] = work["Mileage"].map(_parse_km)

if JUNK_FILTER:
    vb = work["Violation"].astype(str).str.lower()
    harsh_br = vb.str.contains("brak", na=False)
    short_d = work["_ds"].fillna(np.inf) <= MIN_DURATION_SEC
    zero_k = work["_mk"].fillna(0) <= 1e-6
    short_zero = zero_k & (work["_ds"].fillna(np.inf) <= MAX_ZERO_KM_DURATION)
    drop_mask = harsh_br & short_d & zero_k
    drop_mask |= harsh_br & short_zero
    work = work.loc[~drop_mask].copy()

eco_detail_clean = _merge_bursts(work)

eco_summary_raw = _build_summary(eco_detail_raw)
eco_summary_clean = _build_summary(eco_detail_clean)

print(
    f"Rows raw {len(eco_detail_raw)} → clean {len(eco_detail_clean)} "
    f"(merge_gap={MERGE_GAP_SEC}s, junk_filter={JUNK_FILTER})"
)

display(eco_detail_clean.head(15))
display(eco_summary_clean.head(15))

# ========= 3) Excel (raw sheets only) =========
if WRITE_XLSX:

    def save_xlsx(path: Path) -> Path:
        try:
            with pd.ExcelWriter(path, engine="openpyxl") as w:
                eco_detail_raw.to_excel(w, sheet_name="detail_raw", index=False)
                eco_summary_raw.to_excel(w, sheet_name="summary_raw", index=False)
            return path
        except PermissionError:
            alt = path.with_name(path.stem + f"_{datetime.now():%H%M%S}" + path.suffix)
            return save_xlsx(alt)

    saved = save_xlsx(OUT_PATH)
    print("Saved:", saved)

eco_driving_df = eco_detail_clean
eco_driving_summary_df = eco_summary_clean

Rows raw 950 → clean 595 (merge_gap=12s, junk_filter=True)


,Grouping,Violation,Beginning,Initial location,End,Avg. speed,Max. speed,Duration,Mileage,Count,Driver,raw_rows,duration_sum_sec,duration_wall_sec,max_gap_sec
0,FG - Menengai - KAX 536U,,-----,,-----,0 km/h,0 km/h,0:00:00,0.00 km,0,,1,0.0,NaN,0.0
1,FG - Menengai - KBB 161G,,-----,,-----,0 km/h,0 km/h,0:00:00,0.00 km,0,,1,0.0,NaN,0.0
2,FG - Menengai - KBB 880A,,-----,,-----,0 km/h,0 km/h,0:00:00,0.00 km,0,,1,0.0,NaN,0.0
3,FG - Menengai - KBE 726E,,-----,,-----,0 km/h,0 km/h,0:00:00,0.00 km,0,,1,0.0,NaN,0.0
4,FG - Menengai - KBG 388E,,-----,,-----,0 km/h,0 km/h,0:00:00,0.00 km,0,,1,0.0,NaN,0.0
5,FG - Menengai - KBG 389E,,-----,,-----,0 km/h,0 km/h,0:00:00,0.00 km,0,,1,0.0,NaN,0.0
6,FG - Menengai - KBT 114R,,-----,,-----,0 km/h,0 km/h,0:00:00,0.00 km,0,,1,0.0,NaN,0.0
7,FG - Menengai - KBT 116R,FreeWheeling,09.04.2026 01:44:30,"Old Nairobi Road, Kenya, Barnabas Zone A",09.04.2026 01:45:23,37 km/h,42 km/h,0:00:53,0.54 km,1,,1,53.0,53.0,0.0
8,FG - Menengai - KBT 117R,,-----,,-----,0 km/h,0 km/h,0:00:00,0.00 km,0,,1,0.0,NaN,0.0
9,FG - Menengai - KBT 119R,,-----,,-----,0 km/h,0 km/h,0:00:00,0.00 km,0,,1,0.0,NaN,0.0


Violation,Grouping,,Free Wheeling,FreeWheeling,Harsh Acceleration,Harsh Braking,Overrevving,Overspeeding,violations_count,Driver
28,FG - Menengai - KCA 456U,0,0,0,0,371,0,0,371,Karisa Kenga
56,FG - Menengai - KCP 136P,0,0,0,210,0,0,0,210,Philip Muchai
44,FG - Menengai - KCH 502T,0,0,0,0,0,69,0,69,Charles Ngigi
37,FG - Menengai - KCB 345K,0,0,0,0,0,57,0,57,
22,FG - Menengai - KBZ 911V,0,0,0,0,0,33,0,33,
24,FG - Menengai - KCA 224M,0,22,0,0,0,0,6,28,
42,FG - Menengai - KCH 473T,0,0,0,1,0,19,0,20,
60,FG - Menengai - KCT 347W,0,0,0,6,0,0,0,6,Joseph Wanjala
41,FG - Menengai - KCH 467T,0,0,0,0,0,5,0,5,Osero Monari
67,FG - Menengai - KDA 258J,0,0,2,0,0,0,1,3,


Saved: C:\Users\MULINGWA STEPHEN\Downloads\eco_driving_clean_20260510_101644.xlsx


### connecting the files to get whole fleet summary for our overview page ( we can always customize it)

In [ ]:
try:
    from IPython.display import display
except ImportError:
    display = print

# --- pick tables from dfs (edit regex if wrong sheet is chosen) ---
MAIN_PATTERN = r"grouping|last|message|monitor|fleet|unit|online|connection"
EH_PATTERN = r"engine|hour|idl|idle"

WRITE_XLSX = True
OUT_PATH = Path.home() / "Downloads" / f"fleet_summary_{datetime.now():%Y%m%d_%H%M%S}.xlsx"

SORT_BY = ["violations_count", "Max. speed"]  # high → low; edit as needed


def norm_g(s: pd.Series) -> pd.Series:
    return s.astype(str).str.strip()


def parse_max_speed_kmh(sr: pd.Series) -> pd.Series:
    out = []
    for x in sr.astype(str):
        xl = x.lower().replace("km/h", "").replace(",", ".").strip()
        m = re.search(r"[-+]?\d*\.?\d+", xl)
        out.append(float(m.group()) if m else np.nan)
    return pd.Series(out, index=sr.index)


def pick_df(dfs: dict, pattern: str, label: str) -> pd.DataFrame:
    hits = [(k, df) for k, df in dfs.items() if re.search(pattern, str(k), re.I)]
    if not hits:
        raise KeyError(
            f"No table for {label} matching /{pattern}/. Keys:\n"
            + "\n".join(f"  {k!r}" for k in dfs)
        )
    k, df = hits[0]
    print(f"{label}: {k!r} ({len(df)} rows)")
    return df.copy()


def match_rename(df: pd.DataFrame, targets: dict) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    lower = {c.lower(): c for c in df.columns}
    rename = {}
    for canon, aliases in targets.items():
        for a in aliases:
            k = a.strip().lower()
            if k in lower:
                rename[lower[k]] = canon
                break
    return df.rename(columns=rename)


# ---------- checks ----------
if "dfs" not in dir() or not dfs:
    raise RuntimeError("Need dfs")
for x in ("eco_detail_raw", "eco_summary_raw"):
    if x not in dir():
        raise RuntimeError(f"Need {x}")

# ---------- sources ----------
main_df = pick_df(dfs, MAIN_PATTERN, "main")
engine_hours_df = pick_df(dfs, EH_PATTERN, "engine_hours")

main_df = match_rename(
    main_df,
    {
        "Grouping": ["grouping", "unit", "name", "object", "vehicle"],
        "Last message time": ["last message time", "last msg", "last message", "msg time", "time"],
        "Location": ["location", "last location", "position", "place"],
        "Speed": ["speed", "last speed", "current speed"],
        "Driver": ["driver", "driver name"],
    },
)
if "Grouping" not in main_df.columns:
    main_df = main_df.rename(columns={main_df.columns[0]: "Grouping"})

idling_col = None
for cand in ["Idling", "Engine idling", "Idling time", "Idle"]:
    if cand in engine_hours_df.columns:
        idling_col = cand
        break
if idling_col is None:
    for c in engine_hours_df.columns:
        if re.search(r"idl|idle|engine", str(c), re.I):
            idling_col = c
            break
if idling_col is None:
    raise KeyError("No idling column. Have: " + repr(list(engine_hours_df.columns)))

eh = engine_hours_df.rename(columns={idling_col: "Idling"})
eh = match_rename(eh, {"Grouping": ["grouping", "unit", "name", "object", "vehicle"]})
if "Grouping" not in eh.columns:
    eh = eh.rename(columns={eh.columns[0]: "Grouping"})
eh["Grouping"] = norm_g(eh["Grouping"])
eh = eh[["Grouping", "Idling"]].drop_duplicates("Grouping")

viol = eco_summary_raw.copy()
viol["Grouping"] = norm_g(viol["Grouping"])
viol = viol[["Grouping", "violations_count"]].drop_duplicates("Grouping")

ed = eco_detail_raw.copy()
ed["Grouping"] = norm_g(ed["Grouping"])
mx_spd = (
    ed.assign(_mx=parse_max_speed_kmh(ed["Max. speed"]))
    .groupby("Grouping", dropna=False)["_mx"]
    .max()
    .rename("Max. speed")
    .reset_index()
)

m = main_df.copy()
m["Grouping"] = norm_g(m["Grouping"])

summary_fleet = (
    m.merge(viol, on="Grouping", how="left")
    .merge(eh, on="Grouping", how="left")
    .merge(mx_spd, on="Grouping", how="left")
)

COL_ORDER = [
    "Grouping",
    "Last message time",
    "Location",
    "Speed",
    "Driver",
    "violations_count",
    "Idling",
    "Max. speed",
]
summary_fleet = summary_fleet[[c for c in COL_ORDER if c in summary_fleet.columns]]

asc = [False] * len(SORT_BY)
summary_fleet = summary_fleet.sort_values(
    by=[c for c in SORT_BY if c in summary_fleet.columns],
    ascending=[False] * len([c for c in SORT_BY if c in summary_fleet.columns]),
    na_position="last",
    ignore_index=True,
)

display(summary_fleet)

if WRITE_XLSX:
    with pd.ExcelWriter(OUT_PATH, engine="openpyxl") as w:
        summary_fleet.to_excel(w, sheet_name="fleet_summary", index=False)
    print("Saved:", OUT_PATH)

main: 'Unit latest data' (71 rows)
engine_hours: 'Engine hours' (71 rows)


,Grouping,Last message time,Location,Speed,Driver,violations_count,Idling,Max. speed
0,FG - Menengai - KCA 456U,10.05.2026 06:56:12,"Comrades Way, Kakamega, Kenya",0 km/h,Karisa Kenga,453,1:16:38,73.0
1,FG - Menengai - KCP 136P,10.05.2026 07:08:31,"Nakuru-Kisumu Road, Nakuru, Kenya",0 km/h,Philip Muchai,210,1:36:00,77.0
2,FG - Menengai - KCH 502T,10.05.2026 06:25:23,"Pate Road, Nairobi, Kenya",0 km/h,Charles Ngigi,69,2:14:54,70.0
3,FG - Menengai - KCB 345K,07.05.2026 14:12:25,"Nakuru-Kisumu Road, Nakuru, Kenya",0 km/h,,57,0:00:00,81.0
4,FG - Menengai - KBZ 911V,10.05.2026 07:05:18,"Getathuru Road, Kenya, 1.06 km from Nairobi",0 km/h,,33,0:22:04,67.0
...,...,...,...,...,...,...,...,...
66,FG - Menengai - KCW 193E,10.05.2026 06:46:25,"Rongai-Salgaa Road, Kenya, 1.45 km from Rongai",0 km/h,,0,0:00:00,0.0
67,FG - Menengai - KCW 475E,10.05.2026 07:03:26,"Nakuru-Kisumu Road, Nakuru, Kenya",0 km/h,,0,0:00:00,0.0
68,FG - Menengai -KBT 112R,10.05.2026 07:06:05,"Nakuru-Kisumu Road, Nakuru, Kenya",0 km/h,James Kariuki,0,0:00:00,0.0
69,FG - Menengai -KBT 113R,10.05.2026 07:04:51,"Nakuru-Kisumu Road, Nakuru, Kenya",0 km/h,,0,1:20:56,0.0


Saved: C:\Users\MULINGWA STEPHEN\Downloads\fleet_summary_20260510_101654.xlsx
